# Watch a steel ball settle in a tube of water

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Microrobotics-Simulation-Framework/MIME/blob/main/docs/notebooks/sphere_settling_tube.ipynb)

A tiny steel ball, 20 micrometres across (about a quarter of the width of a human hair), is let go in the middle of a
tube of water lying on its side. Gravity pulls it down and the water pushes back, so it sinks slowly and comes to rest
on the bottom of the tube. You will build this little world with MIME, a Python library for simulating microrobots,
run it, plot the fall, and watch it in 3D: the tube, and the ball dropping from its centre to its floor.

Run the cells one by one from the top (click a cell, then press Shift+Enter). Nothing needs installing on your
computer: it all runs on Colab's.

**Install.** This installs MIME, then downloads one data file it needs for this experiment: how the tube's wall
changes the way the water pushes on the ball. Allow a minute or two.

In [ ]:
!pip install -q --pre "mime-engine==0.3.0.dev1"
!mime assets fetch unit_tube_R0.997

**Load the pieces we use.** Every number in MIME carries its unit, so the units are imported too: `um` is
micrometres, `ms` milliseconds.

In [ ]:
import numpy as np

from mime import Body, Experiment, Medium, State, Structure, viz
from mime.dynamics import RigidInertial
from mime.interactions import Contact, Drag, Gravity
from mime.observations import SwimSpeed
from mime.scene import Scene, Sphere, Tube, X, Z
from mime.things import Part
from mime.units import kg_per_m3, m_per_s, m_per_s2, mm_per_s, mPa_s, ms, rad_per_s, um

**The settings.** The numbers you will change later, in "Try it yourself". Density is how heavy a material is for its
size: steel is about eight times as dense as water.

In [ ]:
ball_radius = 10.0     # micrometres
ball_density = 7850.0  # kilograms per cubic metre: steel (water is 998)
run_time = 80.0        # milliseconds

**Start the experiment.** An experiment holds the whole world. Here "up" is the z direction, gravity is Earth's, and
MIME moves time forward in small steps of 2 milliseconds.

In [ ]:
experiment = Experiment("settling_ball", scene=Scene.new(up=Z, units="m"), gravity=m_per_s2(9.81), dt=ms(2.0))

**The tube and the water.** A tube 120 micrometres wide inside, lying along the x direction, filled with water. We
tell MIME the water's density and its viscosity, which is how thick a liquid is (honey's is high, water's low). MIME
asks where each number comes from, so we say.

In [ ]:
tube = experiment.add_structure(Structure(
    "tube", geometry=Tube(axis=+X, inner_radius=um(60.0), wall=um(10.0), length=um(480.0))))
water = experiment.add_medium(Medium("water", fills=tube.lumen))
experiment.declare("water.density", kg_per_m3(998.0), reason="water at 20 degC")
experiment.declare("water.viscosity", mPa_s(1.0), reason="water at 20 degC")

**The ball.** A solid ball that moves as one piece and has mass. It starts at the centre of the tube, not turned
(the orientation `(1, 0, 0, 0)` means "no turn") and not moving.

In [ ]:
ball = experiment.add_body(Body(
    "ball",
    dynamics=RigidInertial(),
    initial=State(position=um((0.0, 0.0, 0.0)), orientation=(1.0, 0.0, 0.0, 0.0),
                  velocity=m_per_s((0.0, 0.0, 0.0)), angular_velocity=rad_per_s((0.0, 0.0, 0.0))),
    parts={"sphere": Part(Sphere(radius=um(ball_radius)))},
))
experiment.declare("ball.parts.sphere.density", kg_per_m3(ball_density), reason="the ball's material")

Now the physics, one piece per cell. If you run one of these cells twice, its piece is added twice and MIME will
refuse to build (it says the same id is declared twice). To fix it, run the cells again from "Start the experiment",
which makes a fresh experiment (or use Runtime, then Run all).

**Gravity.** Gravity pulls the ball down. The water also lifts it a little, the push that makes things float; MIME
adds both.

In [ ]:
experiment.attach(Gravity(), body=ball, medium=water)

**The water's push.** Water pushes back on anything moving through it (this push is called drag), and it pushes
harder close to a wall. MIME works the push out from points spread over the ball's surface, a quarter of its radius
apart, and includes the tube's wall: that is the file we downloaded.

In [ ]:
experiment.attach(Drag.MRS(spacing=um(ball_radius / 4), wall_kind="tube"), body=ball, medium=water)

**The tube's wall keeps the ball in.** When the ball reaches the floor, the wall holds it there, 1.5 micrometres off
the surface.

In [ ]:
experiment.attach(Contact.Projection(margin=um(1.5)), body=ball, surface=tube.inner_surface)

**Run it.** First we tell MIME what to measure: how fast the ball sinks (its speed downward, along -z). Then MIME
checks the experiment and builds it, and runs it for 80 milliseconds, keeping where the ball is at every step
(`history=True`) and a 3D frame every 2 milliseconds for the view. Its display shows it compiling (turning the
experiment into fast code), then stepping through time, then saving the run to a folder, `ball_run`. This takes about
two minutes on Colab.

If the build is refused because the same id is declared twice, a cell above was run twice: run the cells again from
"Start the experiment" (or use Runtime, then Run all).

In [ ]:
sinking = experiment.observe(SwimSpeed(ball, along=-Z))
built = experiment.build()
run = built.run_to("ball_run", ms(run_time), record=[sinking], history=True, stage_every=ms(2.0))
fastest = run.result[sinking].speed.values.to(mm_per_s).max()
print(f"The ball sank at up to {fastest:.2f} millimetres per second.")

**Plot the fall.** The ball's height above the floor of the tube at each moment. MIME kept where the ball was at every
step (its `history`), in metres; the floor is 60 micrometres below the tube's centre, and the bottom of the ball one
radius below the ball's centre. Matplotlib, which Colab has, draws the plot.

In [ ]:
import matplotlib.pyplot as plt

time = run.result.time.to(ms)
centre = np.asarray(run.result.history["ball"]["pose"])[:, 2] * 1e6  # the ball's centre, from metres to micrometres
height = centre + 60.0 - ball_radius

plt.figure(figsize=(8, 4))
plt.plot(time, height, color="#2a78d6", linewidth=2)
plt.xlabel("time (ms)")
plt.ylabel("height above the floor (micrometres)")
plt.title("The ball settling onto the floor of the tube")
plt.grid(color="#e1e0d9")
plt.show()

**Watch it in 3D.** Drag to turn the view, scroll to zoom, and use the play button or the slider to move through time.
The view needs an internet connection.

One thing MIME leaves out for now: right at the floor, the thin film of water squeezed out from under the ball pushes
back hard. That is not modelled yet, so the very last bit of the fall is too fast. It comes in the next development
release, 0.3.0.dev2.

In [ ]:
viz.display("ball_run")

## Try it yourself

Change a number in the settings cell, then run every cell again (Runtime, then Run all).

- **A bigger ball:** `ball_radius = 15.0`. It sinks faster, reaching the floor at about 34 ms instead of about 62 ms:
  a bigger ball's weight grows faster than the water's push on it.
- **A lighter material:** `ball_density = 2700.0` (aluminium) and `run_time = 300.0`. It sinks about four times more
  slowly and lands after about a quarter of a second. The longer run takes a few minutes.

## Learn more

- [MIME's documentation](https://microrobotica.org/mime/0.3.0/): the user guide, its tutorials, and how to read what a run gives back.
- [The example this notebook comes from](https://github.com/Microrobotics-Simulation-Framework/MIME/blob/v0.3.0.dev1/examples/sphere_settling_tube.py), with more of the physics written out.
- The next notebook: [drive a screw-shaped magnetic robot through a tube](https://colab.research.google.com/github/Microrobotics-Simulation-Framework/MIME/blob/main/docs/notebooks/tutorial_a.ipynb).